# Malaria Classification — Revised Methodology (Colab)
Grayscale vs RGB × Frozen vs Fine-tuned, 5 grouped splits, anti-shortcut baselines, Grad-CAM, threshold & calibration analysis, deployment profiling.

**Semua output disimpan otomatis ke Google Drive. Aman terhadap disconnect: jalankan ulang notebook -> run yang selesai dilewati, run yang terputus dilanjutkan.**

Urutan menjalankan: Section 0 -> 1 -> 2 -> 3 -> lalu Section 4 (Eksperimen 1). Setelah itu 5, 6, 7, 8, 9, 10 bisa dijalankan kapan saja (mereka membaca artefak dari Drive).

Tidak ada yang wajib diedit. Anda hanya perlu token Kaggle (`kaggle.json`) saat Section 1.0 memintanya — sekali saja; setelah itu dataset dan token tersimpan di Drive.

## Section 0 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
# ============ EDIT DI SINI (opsional) ============
PROJECT_NAME = "malaria_colorspace"
DRIVE_ROOT   = "/content/drive/MyDrive"
KAGGLE_SLUG  = "iarunava/cell-images-for-detecting-malaria"   # dataset NIH Malaria di Kaggle
DATASET_ZIP  = None   # biarkan None -> otomatis unduh dari Kaggle & cache ke Drive
DATASET_DIR  = None   # isi hanya jika Anda sudah punya folder terekstrak sendiri
# =================================================

PROJ_DIR    = os.path.join(DRIVE_ROOT, PROJECT_NAME)
ART_DIR     = os.path.join(PROJ_DIR, "artifacts")   # model, prediksi, log per-run
SPLIT_DIR   = os.path.join(PROJ_DIR, "splits")      # daftar file per subset (dibekukan)
RESULTS_DIR = os.path.join(PROJ_DIR, "results")     # CSV hasil agregat
LEDGER_PATH = os.path.join(PROJ_DIR, "run_ledger.json")
RESULTS_CSV = os.path.join(RESULTS_DIR, "all_results.csv")
LOCAL_DATA  = "/content/data"                        # salinan lokal citra (ephemeral, bisa dibuat ulang)

for d in [PROJ_DIR, ART_DIR, SPLIT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

if DATASET_ZIP is None:
    DATASET_ZIP = os.path.join(PROJ_DIR, "cell_images.zip")   # lokasi cache dataset di Drive
KAGGLE_JSON_DRIVE = os.path.join(PROJ_DIR, "kaggle.json")     # token Kaggle disimpan agar tak upload ulang

# Hyperparameter
IMG_SIZE = 224
BATCH = 32
MAX_EPOCHS = 50
LR_FROZEN = 1e-4
LR_FINETUNE = 1e-5
FINETUNE_UNFREEZE_FROM = 100   # buka blok atas MobileNetV2 mulai indeks ini
PATIENCE = 5
N_SPLITS = 5
SPLIT_SEEDS = [42, 123, 7, 2024, 99]
POS_LABEL = 1                  # Parasitized = kelas positif; Uninfected = 0
print("Project dir:", PROJ_DIR)

Project dir: /content/drive/MyDrive/malaria_colorspace


In [3]:
import json, time, glob, re, shutil, random, gc, datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, precision_recall_curve,
    roc_curve, brier_score_loss)

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def log(msg):
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)

TF: 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# Catat versi pustaka -> tempel ke bagian Reproducibility di skripsi Anda
import sys, sklearn, scipy, matplotlib
print(sys.version)
for m in [tf, np, pd, sklearn, scipy, matplotlib]:
    print(m.__name__, m.__version__)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
tensorflow 2.20.0
numpy 2.1.3
pandas 2.2.3
sklearn 1.6.1
scipy 1.16.3
matplotlib 3.10.0


## Section 1.0 — Ambil dataset dari Kaggle (sekali) lalu cache ke Drive
Butuh token Kaggle: buka Kaggle > Settings/Account > **Create New API Token** -> unduh `kaggle.json`.
Sel ini meminta upload token **sekali**, menyimpannya ke Drive, mengunduh dataset, lalu **menyimpan zip-nya ke Drive**. Sesi berikutnya langsung memakai cache di Drive — tidak perlu unduh/upload ulang.

In [6]:
import subprocess

# ====== TEMPEL TOKEN KAGGLE ANDA DI SINI ======
KAGGLE_USERNAME = "Malaria"   # <- isi dari kaggle.json ("username")
KAGGLE_KEY      = "KGAT_667e1ced08403e43ce8a00f70f3ab503"   # <- isi dari kaggle.json ("key")
# ==============================================

def setup_kaggle():
    kdir = os.path.expanduser("~/.kaggle"); os.makedirs(kdir, exist_ok=True)
    dst = os.path.join(kdir, "kaggle.json")
    if os.path.exists(dst):                                    # sudah terpasang di sesi ini
        os.chmod(dst, 0o600); log("kaggle.json sudah terpasang."); return
    if os.path.exists(KAGGLE_JSON_DRIVE):                      # tersimpan dari sesi sebelumnya
        shutil.copy(KAGGLE_JSON_DRIVE, dst); os.chmod(dst, 0o600)
        log("kaggle.json dimuat dari Drive."); return
    assert KAGGLE_USERNAME.strip() and KAGGLE_KEY.strip(), \
        "Isi KAGGLE_USERNAME dan KAGGLE_KEY dulu (dari token Kaggle Anda)."
    with open(dst, "w") as f:                                  # buat dari token yang ditempel
        json.dump({"username": KAGGLE_USERNAME.strip(), "key": KAGGLE_KEY.strip()}, f)
    os.chmod(dst, 0o600); shutil.copy(dst, KAGGLE_JSON_DRIVE)  # cache ke Drive untuk sesi berikutnya
    log("kaggle.json dibuat dari token & disalin ke Drive.")

def ensure_dataset_zip():
    if DATASET_DIR: return None                               # pakai folder yang sudah ada
    if os.path.exists(DATASET_ZIP):
        log(f"Dataset zip sudah ada di Drive: {DATASET_ZIP}"); return DATASET_ZIP
    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
    setup_kaggle()
    tmp = "/content/kaggle_dl"; os.makedirs(tmp, exist_ok=True)
    log(f"Mengunduh {KAGGLE_SLUG} dari Kaggle (sekali saja) ...")
    r = subprocess.run(["kaggle","datasets","download","-d",KAGGLE_SLUG,"-p",tmp],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
    zips = glob.glob(os.path.join(tmp, "*.zip"))
    assert zips, "Unduhan gagal - periksa kaggle.json, nama dataset (KAGGLE_SLUG), atau koneksi."
    log(f"Menyalin zip ke Drive (cache permanen): {DATASET_ZIP}")
    shutil.copy(zips[0], DATASET_ZIP)
    return DATASET_ZIP

ensure_dataset_zip()

[14:37:14] kaggle.json dibuat dari token & disalin ke Drive.
[14:37:14] Mengunduh iarunava/cell-images-for-detecting-malaria dari Kaggle (sekali saja) ...
Dataset URL: https://www.kaggle.com/datasets/iarunava/cell-images-for-detecting-malaria
License(s): unknown



  0%|          | 0.00/675M [00:00<?, ?B/s]
  0%|          | 1.00M/675M [00:01<11:37, 1.01MB/s]
  0%|          | 2.00M/675M [00:01<05:58, 1.97MB/s]
  1%|          | 4.00M/675M [00:01<02:49, 4.14MB/s]
  1%|          | 6.00M/675M [00:01<01:45, 6.67MB/s]
  1%|▏         | 9.00M/675M [00:01<01:07, 10.3MB/s]
  2%|▏         | 12.0M/675M [00:01<00:51, 13.6MB/s]
  2%|▏         | 15.0M/675M [00:01<00:43, 16.0MB/s]
  3%|▎         | 18.0M/675M [00:02<00:38, 18.0MB/s]
  3%|▎         | 21.0M/675M [00:02<00:35, 19.2MB/s]
  4%|▎         | 24.0M/675M [00:02<00:34, 19.9MB/s]
  4%|▍         | 27.0M/675M [00:02<00:32, 20.9MB/s]
  4%|▍         | 30.0M/675M [00:02<00:30, 22.1MB/s]
  5%|▍         | 33.0M/675M [00:02<00:31, 21.6MB/s]
  5%|▌         

'/content/drive/MyDrive/malaria_colorspace/cell_images.zip'

## Section 1 — Data: salin lokal, parse pasien, buat & bekukan 5 split

In [7]:
# Latih dari disk lokal (bukan dari Drive yang di-mount) -> jauh lebih cepat & stabil.
def prepare_local_data():
    if os.path.isdir(LOCAL_DATA) and glob.glob(LOCAL_DATA + "/**/*.png", recursive=True):
        log("Data lokal sudah ada."); return
    os.makedirs(LOCAL_DATA, exist_ok=True)
    if not DATASET_DIR and not os.path.exists(DATASET_ZIP):
        ensure_dataset_zip()                 # unduh dari Kaggle bila cache Drive belum ada
    if DATASET_DIR:
        log("Menyalin folder dataset ke lokal ..."); shutil.copytree(DATASET_DIR, LOCAL_DATA, dirs_exist_ok=True)
    elif DATASET_ZIP and os.path.exists(DATASET_ZIP):
        log("Ekstrak ZIP ke lokal ..."); shutil.unpack_archive(DATASET_ZIP, LOCAL_DATA)
    else:
        raise FileNotFoundError("Set DATASET_ZIP atau DATASET_DIR di sel Config.")
    log("Selesai.")

prepare_local_data()

def find_class_dirs(root):
    par = un = None
    for dp, dn, fn in os.walk(root):
        b = os.path.basename(dp).lower()
        if b == "parasitized": par = dp
        if b == "uninfected":  un = dp
    return par, un

par_dir, un_dir = find_class_dirs(LOCAL_DATA)
assert par_dir and un_dir, "Folder Parasitized/Uninfected tidak ditemukan."

rows = []
for label, d in [(1, par_dir), (0, un_dir)]:
    for p in glob.glob(os.path.join(d, "*.png")):
        rows.append((p, label, os.path.basename(p)))
df = pd.DataFrame(rows, columns=["filepath","label","fname"])
df = df[df.fname.str.lower() != "thumbs.db"].reset_index(drop=True)

# ==== GATE PENTING: cek pola nama berkas untuk split tingkat-pasien ====
def parse_group(fn):
    m = re.search(r"(C\d+P\d+)", fn)          # mis. C100P61ThinF...
    if m: return m.group(1)
    m = re.search(r"(IMG_\d+_\d+)", fn)        # fallback: id akuisisi
    if m: return m.group(1)
    return None

df["group"] = df["fname"].apply(parse_group)
print("Contoh nama berkas:", df.fname.head(5).tolist())
n_grouped = df["group"].notna().mean()
log(f"Total citra: {len(df)} | frac parasitized: {df.label.mean():.3f}")
log(f"Nama berkas dengan grup pasien/slide terbaca: {n_grouped:.1%}")
USE_GROUPS = n_grouped > 0.9
log(f"USE_GROUPS = {USE_GROUPS}  ->  {'split grouped (anti-kebocoran pasien)' if USE_GROUPS else 'fallback stratified (akui keterbatasan di skripsi)'}")

[14:38:33] Ekstrak ZIP ke lokal ...
[14:38:41] Selesai.
Contoh nama berkas: ['C184P145ThinF_IMG_20151203_103114_cell_172.png', 'C132P93ThinF_IMG_20151004_151733_cell_141.png', 'C172P133ThinF_IMG_20151119_155307_cell_261.png', 'C39P4thinF_original_IMG_20150622_114122_cell_16.png', 'C51AP12thinF_IMG_20150724_155046_cell_101.png']
[14:38:41] Total citra: 27558 | frac parasitized: 0.500
[14:38:41] Nama berkas dengan grup pasien/slide terbaca: 100.0%
[14:38:41] USE_GROUPS = True  ->  split grouped (anti-kebocoran pasien)


In [8]:
def make_splits():
    if len(sorted(glob.glob(os.path.join(SPLIT_DIR, "split_*.csv")))) >= N_SPLITS:
        log("Split sudah ada di Drive (dibekukan)."); return
    for i, seed in enumerate(SPLIT_SEEDS):
        if USE_GROUPS:
            g = df["group"].values
            tv_idx, te_idx = next(GroupShuffleSplit(1, test_size=0.15, random_state=seed).split(df, df.label, g))
            tv = df.iloc[tv_idx]
            tr_idx, va_idx = next(GroupShuffleSplit(1, test_size=0.1765, random_state=seed).split(tv, tv.label, tv["group"].values))
        else:
            tv_idx, te_idx = next(StratifiedShuffleSplit(1, test_size=0.15, random_state=seed).split(df, df.label))
            tv = df.iloc[tv_idx]
            tr_idx, va_idx = next(StratifiedShuffleSplit(1, test_size=0.1765, random_state=seed).split(tv, tv.label))
        train, val, test = tv.iloc[tr_idx], tv.iloc[va_idx], df.iloc[te_idx]
        assign = pd.concat([train.assign(subset="train"), val.assign(subset="val"), test.assign(subset="test")])
        if USE_GROUPS:  # verifikasi tidak ada pasien yang bocor antar subset
            for a,b in [("train","test"),("train","val"),("val","test")]:
                assert not (set(assign[assign.subset==a].group) & set(assign[assign.subset==b].group)), f"Kebocoran grup {a}-{b}!"
        out = os.path.join(SPLIT_DIR, f"split_{i}_seed{seed}.csv")
        assign[["filepath","label","group","subset"]].to_csv(out, index=False)
        log(f"Simpan {os.path.basename(out)} | train {len(train)} val {len(val)} test {len(test)} | frac parasit uji {test.label.mean():.3f}")

make_splits()

def load_split(i):
    f = sorted(glob.glob(os.path.join(SPLIT_DIR, "split_*.csv")))[i]
    s = pd.read_csv(f)
    return s[s.subset=="train"], s[s.subset=="val"], s[s.subset=="test"]

[14:38:50] Simpan split_0_seed42.csv | train 17576 val 4402 test 5580 | frac parasit uji 0.588
[14:38:50] Simpan split_1_seed123.csv | train 21097 val 3240 test 3221 | frac parasit uji 0.407
[14:38:50] Simpan split_2_seed7.csv | train 20536 val 3648 test 3374 | frac parasit uji 0.457
[14:38:51] Simpan split_3_seed2024.csv | train 19400 val 4900 test 3258 | frac parasit uji 0.390
[14:38:51] Simpan split_4_seed99.csv | train 19214 val 3616 test 4728 | frac parasit uji 0.483


In [10]:
AUTOTUNE = tf.data.AUTOTUNE
GRAY_WEIGHTS = tf.constant([0.2989, 0.5870, 0.1140])  # bobot luminans (dilaporkan di skripsi)

def _gauss_kernel(size, sigma):
    ax = tf.range(-size//2+1, size//2+1, dtype=tf.float32)
    xx, yy = tf.meshgrid(ax, ax)
    k = tf.exp(-(xx**2 + yy**2) / (2.0*sigma**2)); k = k / tf.reduce_sum(k)
    return tf.reshape(k, [size, size, 1, 1])

def gaussian_blur(img):  # hancurkan morfologi, pertahankan warna (baseline B2)
    img = tf.expand_dims(img, 0); k = _gauss_kernel(15, 7.0)
    ch = tf.split(img, 3, axis=-1)
    out = [tf.nn.conv2d(c, k, strides=1, padding="SAME") for c in ch]
    return tf.squeeze(tf.concat(out, axis=-1), 0)

def decode(path):
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, [IMG_SIZE, IMG_SIZE])   # float32 0..255

def to_scheme(img, scheme):
    if scheme == "grayscale":
        g = tf.tensordot(img, GRAY_WEIGHTS, axes=[[-1],[0]])
        img = tf.stack([g, g, g], axis=-1)              # replikasi 3 kanal
    return img

def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.1*255)
    img = tf.image.resize_with_crop_or_pad(img, IMG_SIZE+20, IMG_SIZE+20)
    return tf.image.random_crop(img, [IMG_SIZE, IMG_SIZE, 3])

def make_ds(sub_df, scheme, training, blur=False, batch=BATCH, shuffle_seed=0):
    paths = sub_df.filepath.values; labels = sub_df.label.values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=shuffle_seed, reshuffle_each_iteration=True)
    def _map(path, label):
        img = decode(path)
        if blur: img = gaussian_blur(img)
        img = to_scheme(img, scheme)
        if training: img = augment(img)
        return tf.clip_by_value(img, 0.0, 255.0) / 255.0, label
    return ds.map(_map, num_parallel_calls=AUTOTUNE).batch(batch).prefetch(AUTOTUNE)

## Section 2 — Ledger & penyimpanan (inti anti-kehilangan-data)

In [11]:
def load_ledger():
    if os.path.exists(LEDGER_PATH):
        with open(LEDGER_PATH) as f: return json.load(f)
    return {}

def save_ledger(led):
    tmp = LEDGER_PATH + ".tmp"
    with open(tmp, "w") as f: json.dump(led, f, indent=2)
    os.replace(tmp, LEDGER_PATH)      # tulis atomik

def mark(run_id, status, extra=None):
    led = load_ledger(); led.setdefault(run_id, {})
    led[run_id]["status"] = status
    led[run_id]["updated"] = datetime.datetime.now().isoformat()
    if extra: led[run_id].update(extra)
    save_ledger(led)

def is_done(run_id):
    return load_ledger().get(run_id, {}).get("status") == "done"

def append_result(row):
    d = pd.DataFrame([row])
    d.to_csv(RESULTS_CSV, mode="a", header=not os.path.exists(RESULTS_CSV), index=False)

In [16]:
# Dashboard status — jalankan kapan saja untuk melihat progres
def status():
    led = load_ledger()
    if not led: print("Belum ada run."); return
    s = pd.DataFrame([{"run": k, "status": v.get("status")} for k, v in led.items()])
    print(s.to_string(index=False))
status()

                     run status
exp1_grayscale_fr_split0   done
exp1_grayscale_ft_split0   done
      exp1_rgb_fr_split0   done
      exp1_rgb_ft_split0   done
exp1_grayscale_fr_split1   done
exp1_grayscale_ft_split1   done
      exp1_rgb_fr_split1   done
      exp1_rgb_ft_split1   done
exp1_grayscale_fr_split2   done
exp1_grayscale_ft_split2   done
      exp1_rgb_fr_split2   done
      exp1_rgb_ft_split2   done
exp1_grayscale_fr_split3   done
exp1_grayscale_ft_split3   done
      exp1_rgb_fr_split3   done
      exp1_rgb_ft_split3   done
exp1_grayscale_fr_split4   done
exp1_grayscale_ft_split4   done
      exp1_rgb_fr_split4   done
      exp1_rgb_ft_split4   done


## Section 3 — Model & fungsi training terpadu (dengan resume)

In [13]:
def build_model(finetune=False, seed=0):
    set_seed(seed)
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    if finetune:
        base.trainable = True
        for layer in base.layers[:FINETUNE_UNFREEZE_FROM]:
            layer.trainable = False
    else:
        base.trainable = False
    inp = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)                 # BN tetap mode inferensi (praktik standar)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x); x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x); x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(2, activation="softmax")(x)
    model = tf.keras.Model(inp, out)
    lr = LR_FINETUNE if finetune else LR_FROZEN
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [14]:
def run_one(run_id, split_i, scheme, finetune, train_seed, blur=False, tag="exp1"):
    if is_done(run_id):
        log(f"SKIP {run_id} (sudah selesai)"); return
    run_dir = os.path.join(ART_DIR, run_id); os.makedirs(run_dir, exist_ok=True)
    best_path = os.path.join(run_dir, "best_model.keras")
    latest_w  = os.path.join(run_dir, "latest.weights.h5")
    csvlog    = os.path.join(run_dir, "history.csv")
    mark(run_id, "running", {"split":split_i,"scheme":scheme,"finetune":finetune,
                             "train_seed":train_seed,"blur":blur,"tag":tag})

    tr, va, te = load_split(split_i)
    set_seed(train_seed)
    train_ds = make_ds(tr, scheme, True,  blur=blur, shuffle_seed=train_seed)
    val_ds   = make_ds(va, scheme, False, blur=blur)
    test_ds  = make_ds(te, scheme, False, blur=blur)

    model = build_model(finetune=finetune, seed=train_seed)

    # ---- Resume dari epoch terakhir bila sesi sebelumnya terputus ----
    initial_epoch = 0
    if os.path.exists(csvlog) and os.path.exists(latest_w):
        try:
            initial_epoch = int(pd.read_csv(csvlog)["epoch"].max()) + 1
            model.load_weights(latest_w)
            log(f"RESUME {run_id} dari epoch {initial_epoch}")
        except Exception as e:
            log(f"Resume gagal ({e}); mulai dari awal"); initial_epoch = 0

    cbs = [
        tf.keras.callbacks.ModelCheckpoint(best_path, monitor="val_loss",
            save_best_only=True, save_weights_only=False),               # model terbaik -> Drive
        tf.keras.callbacks.ModelCheckpoint(latest_w, save_best_only=False,
            save_weights_only=True, save_freq="epoch"),                  # snapshot tiap epoch -> Drive
        tf.keras.callbacks.CSVLogger(csvlog, append=(initial_epoch>0)),  # log tiap epoch -> Drive
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
    ]
    model.fit(train_ds, validation_data=val_ds, epochs=MAX_EPOCHS,
              initial_epoch=initial_epoch, callbacks=cbs, verbose=1)

    if os.path.exists(best_path):
        model = tf.keras.models.load_model(best_path)

    # Simpan probabilitas val & test -> Eksperimen 3/7 tak perlu latih ulang
    p_val = model.predict(val_ds,  verbose=0)[:, POS_LABEL]
    p_te  = model.predict(test_ds, verbose=0)[:, POS_LABEL]
    y_val = va.label.values; y_te = te.label.values
    np.savez(os.path.join(run_dir, "preds.npz"), y_val=y_val, p_val=p_val, y_test=y_te, p_test=p_te)

    yhat = (p_te >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, yhat).ravel()
    row = {"run_id":run_id,"tag":tag,"split":split_i,"scheme":scheme,"finetune":finetune,
           "train_seed":train_seed,"blur":blur,
           "accuracy":accuracy_score(y_te,yhat),
           "precision":precision_score(y_te,yhat,pos_label=POS_LABEL,zero_division=0),
           "recall":recall_score(y_te,yhat,pos_label=POS_LABEL),
           "f1":f1_score(y_te,yhat,pos_label=POS_LABEL),
           "auc":roc_auc_score(y_te,p_te),
           "brier":brier_score_loss((y_te==POS_LABEL).astype(int),p_te),
           "n_test":len(y_te),"tn":tn,"fp":fp,"fn":fn,"tp":tp}
    append_result(row)
    mark(run_id, "done", {"metrics":{k:float(row[k]) for k in ["accuracy","precision","recall","f1","auc","brier"]}})
    log(f"DONE {run_id}: acc {row['accuracy']:.4f} auc {row['auc']:.4f} recall {row['recall']:.4f} fn {fn}")
    del model; gc.collect(); tf.keras.backend.clear_session()

## Section 4 — Eksperimen 1: faktorial 2×2 (warna × transfer) × 5 split (menjawab RQ1)
Jalankan sel ini berkali-kali bila perlu; run yang sudah selesai otomatis dilewati.

In [15]:
TRAIN_SEED = 42   # variasi antar-split adalah sumber ketidakpastian utama
for split_i in range(N_SPLITS):
    for scheme in ["grayscale", "rgb"]:
        for finetune in [False, True]:
            ft = "ft" if finetune else "fr"
            run_one(f"exp1_{scheme}_{ft}_split{split_i}", split_i, scheme, finetune, TRAIN_SEED, tag="exp1")
status()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/50
550/550 ━━━━━━━━━━━━━━━━━━━━ 68s 87ms/step - accuracy: 0.8136 - loss: 0.4080 - val_accuracy: 0.8789 - val_loss: 0.2986
Epoch 2/50
550/550 ━━━━━━━━━━━━━━━━━━━━ 22s 39ms/step - accuracy: 0.8832 - loss: 0.2822 - val_accuracy: 0.8580 - val_loss: 0.3231
Epoch 3/50
550/550 ━━━━━━━━━━━━━━━━━━━━ 21s 38ms/step - accuracy: 0.8966 - loss: 0.2530 - val_accuracy: 0.8660 - val_loss: 0.3058
Epoch 4/50
550/550 ━━━━━━━━━━━━━━━━━━━━ 21s 39ms/step - accuracy: 0.9045 - loss: 0.2384 - val_accuracy: 0.9025 - val_loss: 0.2429
Epoch 5/50
550/550 ━━━━━━━━━━━━━━━━━━━━ 21s 38ms/step - accuracy: 0.9075 - loss: 0.2253 - val_accuracy: 0.9069 - val_loss: 0.2279
Epoch 6/50
550/550 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - accuracy: 0.9133 - loss: 0.2167 - val_accuracy: 0.9041 - val_loss: 0.2378
Epoch 7/50
550/550 ━━━━━━━━━━━━━━━━━━━━ 21s 38ms/step - accuracy: 0.9191 - loss: 0.2092 - val_accuracy: 0.9019 - val_loss: 0.2402
Epoch 8/50
550/550 ━━━━━━━━━━━━━━━━━━━━ 2

## Section 5 — Baseline anti-jalan-pintas (mempertahankan RQ1)
Jika baseline ini sudah akurat, itu bukti ada shortcut warna/artefak di dataset — wajib Anda ketahui sebelum menyimpulkan.

In [17]:
# B1 — hanya fitur warna (rerata/simpangan/histogram kanal). Tidak ada morfologi sama sekali.
def color_features(paths):
    feats = []
    for p in paths:
        img = (tf.image.resize(tf.io.decode_png(tf.io.read_file(p), channels=3), [64,64]) / 255.0).numpy().reshape(-1,3)
        hist = np.concatenate([np.histogram(img[:,c], bins=8, range=(0,1))[0] for c in range(3)]).astype(float)
        feats.append(np.concatenate([img.mean(0), img.std(0), hist/hist.sum()]))
    return np.array(feats)

def run_baseline_color(split_i):
    run_id = f"baseline_color_split{split_i}"
    if is_done(run_id): log(f"SKIP {run_id}"); return
    mark(run_id, "running", {"tag":"baseline_color","split":split_i})
    tr, va, te = load_split(split_i); cache = os.path.join(ART_DIR, run_id); os.makedirs(cache, exist_ok=True)
    Xtr, Xte = color_features(tr.filepath.values), color_features(te.filepath.values)
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=1000).fit(sc.transform(Xtr), tr.label.values)
    p_te = clf.predict_proba(sc.transform(Xte))[:, POS_LABEL]; y_te = te.label.values
    yhat = (p_te>=0.5).astype(int)
    np.savez(os.path.join(cache,"preds.npz"), y_test=y_te, p_test=p_te)
    tn,fp,fn,tp = confusion_matrix(y_te,yhat).ravel()
    row = {"run_id":run_id,"tag":"baseline_color","split":split_i,"scheme":"coloronly","finetune":False,
           "train_seed":42,"blur":False,"accuracy":accuracy_score(y_te,yhat),
           "precision":precision_score(y_te,yhat,pos_label=POS_LABEL,zero_division=0),
           "recall":recall_score(y_te,yhat,pos_label=POS_LABEL),"f1":f1_score(y_te,yhat,pos_label=POS_LABEL),
           "auc":roc_auc_score(y_te,p_te),"brier":brier_score_loss((y_te==POS_LABEL).astype(int),p_te),
           "n_test":len(y_te),"tn":tn,"fp":fp,"fn":fn,"tp":tp}
    append_result(row); mark(run_id,"done",{"metrics":{"auc":float(row["auc"]),"accuracy":float(row["accuracy"])}})
    log(f"DONE {run_id}: acc {row['accuracy']:.4f} auc {row['auc']:.4f}  <- tinggi = ada shortcut warna")

for split_i in range(N_SPLITS):
    run_baseline_color(split_i)

[18:01:37] DONE baseline_color_split0: acc 0.8016 auc 0.8913  <- tinggi = ada shortcut warna
[18:02:59] DONE baseline_color_split1: acc 0.8708 auc 0.9081  <- tinggi = ada shortcut warna
[18:04:20] DONE baseline_color_split2: acc 0.7967 auc 0.8562  <- tinggi = ada shortcut warna
[18:05:39] DONE baseline_color_split3: acc 0.8072 auc 0.8349  <- tinggi = ada shortcut warna
[18:07:00] DONE baseline_color_split4: acc 0.7936 auc 0.8434  <- tinggi = ada shortcut warna


In [ ]:
# B2 — MobileNetV2 (RGB, frozen) pada citra yang di-blur berat: morfologi hilang, warna tetap.
for split_i in range(N_SPLITS):
    run_one(f"baseline_blur_split{split_i}", split_i, "rgb", False, 42, blur=True, tag="baseline_blur")
status()

[23:39:48] SKIP baseline_blur_split0 (sudah selesai)
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[23:39:56] RESUME baseline_blur_split1 dari epoch 17
Epoch 18/50


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


660/660 ━━━━━━━━━━━━━━━━━━━━ 392s 566ms/step - accuracy: 0.9461 - loss: 0.1510 - val_accuracy: 0.9435 - val_loss: 0.1543
Epoch 19/50
660/660 ━━━━━━━━━━━━━━━━━━━━ 355s 538ms/step - accuracy: 0.9445 - loss: 0.1506 - val_accuracy: 0.9392 - val_loss: 0.1506
Epoch 20/50
660/660 ━━━━━━━━━━━━━━━━━━━━ 353s 534ms/step - accuracy: 0.9458 - loss: 0.1491 - val_accuracy: 0.9457 - val_loss: 0.1515
Epoch 21/50
660/660 ━━━━━━━━━━━━━━━━━━━━ 355s 538ms/step - accuracy: 0.9472 - loss: 0.1451 - val_accuracy: 0.9395 - val_loss: 0.1558
Epoch 22/50
660/660 ━━━━━━━━━━━━━━━━━━━━ 345s 523ms/step - accuracy: 0.9482 - loss: 0.1450 - val_accuracy: 0.9438 - val_loss: 0.1524
Epoch 23/50
660/660 ━━━━━━━━━━━━━━━━━━━━ 347s 525ms/step - accuracy: 0.9474 - loss: 0.1428 - val_accuracy: 0.9454 - val_loss: 0.1502
Epoch 24/50
660/660 ━━━━━━━━━━━━━━━━━━━━ 345s 523ms/step - accuracy: 0.9484 - loss: 0.1431 - val_accuracy: 0.9466 - val_loss: 0.1431
Epoch 25/50
660/660 ━━━━━━━━━━━━━━━━━━━━ 350s 530ms/step - accuracy: 0.9486 - los

## Section 6 — Eksperimen 3: analisis ambang (menjawab RQ2)
Ambang dipilih pada VALIDASI (maksimum F1), dilaporkan pada UJI. Tidak ada kebocoran test.

In [ ]:
res = pd.read_csv(RESULTS_CSV); exp1 = res[res.tag=="exp1"]
best_scheme, best_ft = exp1.groupby(["scheme","finetune"])["auc"].mean().idxmax()
ft = "ft" if best_ft else "fr"
log(f"Konfigurasi terbaik (AUC rata-rata): scheme={best_scheme}, finetune={best_ft}")

def load_preds(run_id): return np.load(os.path.join(ART_DIR, run_id, "preds.npz"))

thr_rows = []
for split_i in range(N_SPLITS):
    d = load_preds(f"exp1_{best_scheme}_{ft}_split{split_i}")
    prec, rec, thr = precision_recall_curve((d["y_val"]==POS_LABEL).astype(int), d["p_val"])
    f1s = 2*prec*rec/(prec+rec+1e-9)
    best_t = float(thr[np.nanargmax(f1s[:-1])]) if len(thr) else 0.5   # dipilih di validasi
    y, p = d["y_test"], d["p_test"]; yopt = (p>=best_t).astype(int)
    thr_rows.append({"split":split_i,"threshold_val":best_t,
        "recall_0.5":recall_score(y,(p>=0.5).astype(int),pos_label=POS_LABEL),
        "recall_opt":recall_score(y,yopt,pos_label=POS_LABEL),
        "precision_opt":precision_score(y,yopt,pos_label=POS_LABEL,zero_division=0),
        "fn_0.5":int(((p<0.5)&(y==POS_LABEL)).sum()),
        "fn_opt":int(((p<best_t)&(y==POS_LABEL)).sum())})
thr_df = pd.DataFrame(thr_rows); thr_df.to_csv(os.path.join(RESULTS_DIR,"exp3_threshold.csv"), index=False)
print(thr_df.to_string(index=False))

## Section 7 — Kalibrasi & precision pada prevalensi klinis rendah

In [ ]:
def precision_at_prevalence(y, p, thr, pi):
    y = (y==POS_LABEL).astype(int); yhat = (p>=thr).astype(int)
    tpr = ((yhat==1)&(y==1)).sum()/max((y==1).sum(),1)
    fpr = ((yhat==1)&(y==0)).sum()/max((y==0).sum(),1)
    return (tpr*pi)/(tpr*pi + fpr*(1-pi) + 1e-9)

cal_rows = []
for split_i in range(N_SPLITS):
    d = load_preds(f"exp1_{best_scheme}_{ft}_split{split_i}"); y, p = d["y_test"], d["p_test"]
    cal_rows.append({"split":split_i,"brier":brier_score_loss((y==POS_LABEL).astype(int),p),
        "prec@50%":precision_at_prevalence(y,p,0.5,0.50),
        "prec@5%": precision_at_prevalence(y,p,0.5,0.05),
        "prec@1%": precision_at_prevalence(y,p,0.5,0.01)})
cal_df = pd.DataFrame(cal_rows); cal_df.to_csv(os.path.join(RESULTS_DIR,"exp3b_calibration.csv"), index=False)
print(cal_df.round(4).to_string(index=False))

## Section 8 — Eksperimen 2: Grad-CAM (interpretabilitas)
Jalankan juga pada baseline_blur untuk melihat "seperti apa rupa shortcut" — pembanding yang jauh lebih meyakinkan daripada narasi kualitatif.

In [ ]:
import matplotlib.pyplot as plt
def gradcam_heatmap(model, x):   # x: (1,224,224,3) di [0,1]
    base = next(l for l in model.layers if isinstance(l, tf.keras.Model))
    last_conv = next(l.name for l in reversed(base.layers) if isinstance(l, tf.keras.layers.Conv2D))
    conv_model = tf.keras.Model(base.input, base.get_layer(last_conv).output)
    head = model.layers[model.layers.index(base)+1:]
    with tf.GradientTape() as tape:
        conv_out = conv_model(x); tape.watch(conv_out)
        z = tf.keras.layers.GlobalAveragePooling2D()(conv_out)
        for l in head:
            if isinstance(l, (tf.keras.layers.GlobalAveragePooling2D, tf.keras.layers.Dropout)): continue
            z = l(z)
        score = z[:, POS_LABEL]
    grads = tape.gradient(score, conv_out)
    w = tf.reduce_mean(grads, axis=(0,1,2))
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0]*w, axis=-1))
    return (cam/(tf.reduce_max(cam)+1e-9)).numpy(), last_conv

def show_gradcam(run_id, n=4):
    model = tf.keras.models.load_model(os.path.join(ART_DIR, run_id, "best_model.keras"))
    _, _, te = load_split(0)
    scheme = "grayscale" if "grayscale" in run_id else "rgb"
    blur = run_id.startswith("baseline_blur")
    sample = te[te.label==POS_LABEL].head(n)
    fig, ax = plt.subplots(2, n, figsize=(3*n, 6))
    for j, (_, r) in enumerate(sample.iterrows()):
        img = decode(r.filepath)
        if blur: img = gaussian_blur(img)
        img = tf.clip_by_value(to_scheme(img, scheme),0,255)/255.0
        cam, lc = gradcam_heatmap(model, tf.expand_dims(img,0))
        cam = tf.image.resize(cam[...,None], [IMG_SIZE,IMG_SIZE]).numpy().squeeze()
        ax[0,j].imshow(img.numpy()); ax[0,j].axis("off")
        ax[1,j].imshow(img.numpy()); ax[1,j].imshow(cam, cmap="jet", alpha=0.5); ax[1,j].axis("off")
    ax[0,0].set_ylabel("input"); fig.suptitle(f"Grad-CAM: {run_id} (last_conv={lc})")
    out = os.path.join(RESULTS_DIR, f"gradcam_{run_id}.png"); fig.savefig(out, bbox_inches="tight", dpi=120)
    plt.show(); log(f"Disimpan {out}")
    del model; tf.keras.backend.clear_session()

show_gradcam(f"exp1_{best_scheme}_{ft}_split0")
# show_gradcam("baseline_blur_split0")   # buka untuk melihat pola shortcut

## Section 9 — Eksperimen 4: profil kelayakan penerapan (menjawab RQ3)

In [ ]:
prof = []
for scheme in ["grayscale","rgb"]:
    for finetune in [False, True]:
        ft2 = "ft" if finetune else "fr"; mp = os.path.join(ART_DIR, f"exp1_{scheme}_{ft2}_split0", "best_model.keras")
        if not os.path.exists(mp): continue
        m = tf.keras.models.load_model(mp); dummy = tf.random.uniform((1,IMG_SIZE,IMG_SIZE,3))
        for _ in range(5): m(dummy)                       # warmup
        t = time.time()
        for _ in range(50): m(dummy)
        prof.append({"scheme":scheme,"finetune":finetune,"size_mb":round(os.path.getsize(mp)/1e6,2),
                     "params":int(m.count_params()),"latency_ms":round((time.time()-t)/50*1000,2)})
        del m; tf.keras.backend.clear_session()
prof_df = pd.DataFrame(prof); prof_df.to_csv(os.path.join(RESULTS_DIR,"exp4_deployment.csv"), index=False)
print(prof_df.to_string(index=False))

## Section 10 — Agregasi & statistik (paired test + CI + effect size)

In [ ]:
from scipy import stats
res = pd.read_csv(RESULTS_CSV); exp1 = res[res.tag=="exp1"]

summ = exp1.groupby(["scheme","finetune"]).agg(
    acc_m=("accuracy","mean"), acc_s=("accuracy","std"),
    auc_m=("auc","mean"),      auc_s=("auc","std"),
    rec_m=("recall","mean"),   rec_s=("recall","std")).round(4)
print("== Ringkasan mean +/- SD antar-split =="); print(summ, "\n")

def paired(metric, finetune):   # RGB vs grayscale, berpasangan per-split
    a = exp1[(exp1.scheme=="rgb")&(exp1.finetune==finetune)].sort_values("split")[metric].values
    b = exp1[(exp1.scheme=="grayscale")&(exp1.finetune==finetune)].sort_values("split")[metric].values
    diff = a-b; t,p = stats.ttest_rel(a,b)
    ci = stats.t.interval(0.95, len(diff)-1, loc=diff.mean(), scale=stats.sem(diff))
    d = diff.mean()/(diff.std(ddof=1)+1e-9)
    return {"metric":metric,"finetune":finetune,"mean_diff_rgb_minus_gray":round(float(diff.mean()),4),
            "ci_low":round(float(ci[0]),4),"ci_high":round(float(ci[1]),4),
            "cohen_d":round(float(d),3),"t":round(float(t),3),"p":round(float(p),4)}

stat_df = pd.DataFrame([paired(m,ft) for m in ["accuracy","auc","recall"] for ft in [False,True]])
stat_df.to_csv(os.path.join(RESULTS_DIR,"exp1_stats.csv"), index=False)
print("== Paired RGB vs Grayscale (per level transfer) =="); print(stat_df.to_string(index=False), "\n")

# Baseline vs model penuh (bukti ada/tidaknya shortcut)
for tag in ["baseline_color","baseline_blur"]:
    sub = res[res.tag==tag]
    if len(sub): print(f"{tag}: acc {sub.accuracy.mean():.4f} +/- {sub.accuracy.std():.4f} | auc {sub.auc.mean():.4f}")